In [0]:
%run ../init_framework_gold

In [0]:
# --- 1. SOURCE INGESTION (Gold Layer) ---

# Read Defaulters Inquiry Silver Table
df_inquiry = read_delta_stream(
    spark=spark, 
    table_name=DEFAULTERS_INQUIRY_SILVER
)

# Read Defaulters Delinquents Silver Table
df_delinq = read_delta_stream(
    spark=spark, 
    table_name=DEFAULTERS_DELINQ_SILVER
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

# Configured for 8-core cluster (16 shuffle partitions).
spark.conf.set("spark.sql.shuffle.partitions", "16")

# --- 2. STREAMING JOIN & SELECTION ---
# Joining Inquiry (i) and Delinquency (d) on member_id
df_joined = (
    df_inquiry.alias("i")
    .join(df_delinq.alias("d"), "member_id", "inner")
    .select(
        col("i.member_id"),
        col("i.pub_rec"),
        col("i.pub_rec_bankruptcies"),
        col("i.inq_last_6mths"),
        col("d.delinq_2yrs"),
        # Metadata aliasing for the 'Truth' filter in foreachBatch
        col("i._change_type").alias("inquiry_change_type"),
        col("d._change_type").alias("delinq_change_type"),
    )
)

In [0]:
# --- 3. APPLY GOLD METADATA ---
# # Injecting lineage and timestamp
df_with_metadata = apply_gold_metadata(df_joined, lit("calc_defaulters_performance"))

In [0]:
# --- 4. SINK LOGIC (foreachBatch) ---
def upsert_defaulters_performance_to_gold(microBatchDF, batchId):
    """
    Stateless foreachBatch implementation for Gold Layer.
    Handles CDF metadata filtering, multi-category scoring via lookup, and Delta Merge.
    """
    spark_session = microBatchDF.sparkSession

    # 1. Filter for 'Latest Truth'
    # Ensures we process only valid events and handles State Store NULL matches
    df_clean = microBatchDF.filter(
        (col("inquiry_change_type").isin("insert", "update_postimage"))
        & (
            col("delinq_change_type").isin("insert", "update_postimage")
            | col("delinq_change_type").isNull()
        )
    )

    # 2. Register Source View for SQL transformations
    df_clean.createOrReplaceTempView("batch_updates")

    # 3. Define Scoring Logic (Categorization Keys)
    scoring_query = f""" 
        SELECT 
            member_id,
            -- 1. Delinquency Category
            CASE 
                WHEN delinq_2yrs = 0 THEN 'EXCELLENT'
                WHEN delinq_2yrs BETWEEN 1 AND 2 THEN 'BAD'
                WHEN delinq_2yrs BETWEEN 3 AND 5 THEN 'VERY_BAD'
                ELSE 'UNACCEPTABLE' -- Covers > 5 and NULL
            END AS delinq_key,
            
            -- 2. Public Records Category
            CASE 
                WHEN pub_rec = 0 THEN 'EXCELLENT'
                WHEN pub_rec BETWEEN 1 AND 2 THEN 'BAD'
                ELSE 'VERY_BAD' -- Covers > 2 and NULL
            END AS public_records_key,
            
            -- 3. Public Bankruptcies Category
            CASE 
                WHEN pub_rec_bankruptcies = 0 THEN 'EXCELLENT'
                WHEN pub_rec_bankruptcies BETWEEN 1 AND 2 THEN 'BAD'
                ELSE 'VERY_BAD' -- Covers > 2 and NULL
            END AS public_bankruptcies_key,
            
            -- 4. Inquiries Category
            CASE 
                WHEN inq_last_6mths = 0 THEN 'EXCELLENT'
                WHEN inq_last_6mths BETWEEN 1 AND 2 THEN 'BAD'
                WHEN inq_last_6mths BETWEEN 3 AND 5 THEN 'VERY_BAD'
                ELSE 'UNACCEPTABLE' -- Covers > 5 and NULL
            END AS enq_key,
            _updated_at,
            _scoring_module
        FROM batch_updates       
    """

    # 4. Create Scored Keys View
    df_scored = spark_session.sql(scoring_query)
    df_scored.createOrReplaceTempView("scored_updates")

    # 5. Map Keys to points via Broadcast Join with Reference Mapping
    defaulter_query = f"""
        SELECT 
            perf.member_id,
            r1.points AS delinq_pts,
            r2.points AS public_records_pts,
            r3.points AS public_bankruptcies_pts,
            r4.points AS inquiry_pts,
            perf._updated_at,
            perf._scoring_module
        FROM scored_updates perf 
        JOIN lending_risk.silver.ref_score_points_mapping r1
            ON perf.delinq_key = r1.rating_key
        JOIN lending_risk.silver.ref_score_points_mapping r2
            ON perf.public_records_key = r2.rating_key
        JOIN lending_risk.silver.ref_score_points_mapping r3
            ON perf.public_bankruptcies_key = r3.rating_key
        JOIN lending_risk.silver.ref_score_points_mapping r4
            ON perf.enq_key = r4.rating_key
    """

    # 6. Final Preparation for Merge
    df_defaulters = spark_session.sql(defaulter_query)    
    df_defaulters.createOrReplaceTempView("defaulters_updates")

    # 7. Execute Idempotent Merge into Gold Layer
    spark_session.sql(f"""
        MERGE INTO {CATALOG}.gold.calc_defaulters_performance AS target
        USING defaulters_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED THEN
          UPDATE SET 
            target.delinq_pts = source.delinq_pts,
            target.public_records_pts = source.public_records_pts,
            target.public_bankruptcies_pts = source.public_bankruptcies_pts,
            target.inquiry_pts = source.inquiry_pts,
            target._updated_at = source._updated_at,
            target._scoring_module = source._scoring_module
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, 
            delinq_pts, 
            public_records_pts, 
            public_bankruptcies_pts, 
            inquiry_pts, 
            _updated_at, 
            _scoring_module
          )
          VALUES (
            source.member_id, 
            source.delinq_pts, 
            source.public_records_pts, 
            source.public_bankruptcies_pts, 
            source.inquiry_pts, 
            source._updated_at, 
            source._scoring_module
          )
    """)

# --- STREAM TRIGGER ---
# Using availableNow=True for batch-like cost efficiency on a streaming engine
query = (
    df_with_metadata.writeStream
    .outputMode("append")
    .option("checkpointLocation", GOLD_CHECKPOINT_DEFAULTERS_PERF)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_performance_to_gold)
    .start()
)

# Block notebook termination until micro-batch is committed
query.awaitTermination()

In [0]:
# # Forcefully clear the checkpoint directory to reset stream state
# dbutils.fs.rm(GOLD_CHECKPOINT_DEFAULTERS_PERF, recurse=True)

# print(f"Purged: {GOLD_CHECKPOINT_DEFAULTERS_PERF}")

In [0]:
%sql
select * from lending_risk.gold.calc_defaulters_performance limit 3;